# DGM-HMamba — demo

<a href="https://colab.research.google.com/github/zarrinmahdavipour/DGM-HMamba/blob/main/notebooks/DGM_HMamba_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Domain-adaptive Diffusion-Guided Hierarchical Mamba for cross-domain few-shot hyperspectral image classification.**

This notebook runs the code end to end:

1. **Smoke tests on synthetic data** – one forward pass and a few training steps of both stages.
2. **Real-data demo, Indian Pines → Pavia University** – downloads the two public scenes, runs Stage 1 (source pre-training with the domain-adversarial diffusion module) and Stage 2 (episodic meta-training on the target meta-training pool), evaluates 5-way 1-shot and 5-shot episodes on the held-out meta-testing pool, and draws a classification map.

**Runtime:** about 5–10 minutes on a Colab GPU (*Runtime → Change runtime type → T4 GPU*); it also runs on CPU, more slowly.

> The demo uses a small model (≈0.3 M parameters) and a short schedule so that it finishes quickly. Its accuracies show that the pipeline works; they are **not** the results reported in the manuscript. See the README for the full training command and the current status of the implementation.

## 0. Setup

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/zarrinmahdavipour/DGM-HMamba.git"

def run(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"Step failed (exit code {proc.returncode}): {cmd}\n"
                           "The error message is in the output above.")

if os.path.basename(os.getcwd()) == "notebooks":      # opened from a local clone
    os.chdir("..")
if not os.path.isdir("dgm_hmamba"):                     # Colab session
    if not os.path.isdir("DGM-HMamba"):
        run(f"git clone -q {REPO_URL}")
    os.chdir("DGM-HMamba")
run("git pull -q || true")                              # always use the latest code

run("pip -q install -r requirements.txt")
print("Working directory:", os.getcwd())

In [ ]:
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Smoke tests on synthetic data (≈1 min)

`demo_forward.py` evaluates the Stage-1 losses (L_cls, L_diff, L_adv) and one Stage-2 episode (L_epi, L_gcl) of a tiny model. At initialisation L_adv ≈ 2·ln 2 ≈ 1.39 and L_epi ≈ ln N, the chance-level values.

In [ ]:
run("python scripts/demo_forward.py")

`demo_train.py` runs a few real optimisation steps of Stage 1 and Stage 2 (random data, so accuracy stays at chance; this only checks that gradients flow through both stages).

In [ ]:
run("python scripts/demo_train.py")

## 2. Real data: Indian Pines → Pavia University

Download the two scenes from the public EHU repository (≈40 MB) and check that they load.

In [ ]:
run("python scripts/prepare_data.py --data-root data --download indian_pines pavia_u")

Run the demo. The source is Indian Pines (AVIRIS, 200 bands, 16 classes) and the target is Pavia University (ROSIS, 103 bands, 9 classes); per-sensor 1×1 band projections map both to the shared 200-channel space.

Useful options: `--stage1-iters`, `--stage2-steps` (training length), `--k-shot`, `--n-way`, `--no-map` (skip the map), `--device cpu`.

In [ ]:
run("python scripts/demo_real.py --data-root data --output-dir runs/demo")

### Results

In [ ]:
import json, os
from IPython.display import Image, display

if not os.path.isfile("runs/demo/demo_summary.json"):
    raise RuntimeError("The demo did not finish, so there are no results yet. "
                       "Scroll up to the first cell with an error message.")

display(Image("runs/demo/demo_maps.png"))
display(Image("runs/demo/demo_curves.png"))

summary = json.load(open("runs/demo/demo_summary.json"))
print(f"Demo model: {summary['trainable_parameters']/1e6:.2f} M parameters on {summary['device']}")
for name, m in summary["episodic_metrics"].items():
    print(f"{name:>12}: OA {m['oa_mean']*100:5.2f} ± {m['oa_std']*100:4.2f} %   "
          f"AA {m['aa']*100:5.2f} %   kappa {m['kappa']:.4f}   ({m['n_episodes']} episodes)")
if summary["map_metrics"]:
    mm = summary["map_metrics"]
    print(f"{'map':>12}: OA {mm['oa']*100:5.2f} %   AA {mm['aa']*100:5.2f} %   kappa {mm['kappa']:.4f}")
print({k: round(v) for k, v in summary["timings_s"].items()}, "seconds")

## 3. Using the library directly

The model can also be used from Python. `encode` runs the diffusion enhancement (DADGE) followed by the hierarchical dual-path Mamba encoder (HDPM); `predict_episode` builds and evolves class prototypes (DSSPE) from a labelled support set and classifies the queries.

In [ ]:
import torch
from dgm_hmamba import DGMHMamba, get_default_config

cfg = get_default_config()            # manuscript hyper-parameters (see dgm_hmamba/config.py)
cfg.hdpm.widths = (32, 32, 32)        # shrink for a quick CPU example
cfg.hdpm.blocks_per_level, cfg.hdpm.state_dim, cfg.hdpm.expand_factor = 1, 4, 1
cfg.dadge.T = cfg.dadge.default_inference_steps = 10

model = DGMHMamba(cfg, source_bands=200, target_bands=103, num_source_classes=16).eval()

n_way, k_shot, n_query = 3, 5, 4
support = torch.rand(n_way * k_shot, 103, 9, 9)          # raw Pavia-U-like patches
query = torch.rand(n_query, 103, 9, 9)
support_y = torch.arange(n_way).repeat_interleave(k_shot)
anchors = torch.randn(n_way, 300)                        # placeholder class-name embeddings

with torch.no_grad():
    sx0 = model.project_bands(support, is_source=False)  # 103 -> 200 channels
    qx0 = model.project_bands(query, is_source=False)
    f = model.encode(qx0)                                # full T-step DDPM chain
    f_fast = model.encode(qx0, num_inference_steps=5, ddim=True)   # accelerated DDIM
    pred = model.predict_episode(sx0, support_y, qx0, n_way, anchors)
print("embedding:", tuple(f.shape), "| predictions:", pred.tolist())

## 4. Full training

`scripts/train.py` runs the complete two-stage protocol with logging, checkpoints, an optional validation pool carved from the target meta-training pool, and a single final evaluation on the meta-testing pool:

```bash
python scripts/train.py --source indian_pines --target pavia_u --data-root data \
    --n-way 5 --k-shot 5 --q-query 15 --stage1-epochs 200 --stage2-steps 100 \
    --seed 42 --output-dir runs/ip2pu_5shot_s42
```

The manuscript-size model needs a large GPU because the selective scan in `mamba_block.py` is a plain PyTorch loop; use `--preset reduced` or `--preset tiny` on smaller hardware. See the README for details and the list of known differences from the manuscript.